# 03 — Lexical Retrieval Lab: Inverted Index → TF → IDF → BM25 → Dense Search

**Goal:** Understand the mechanics of keyword search, implement BM25 from scratch, and compare its rankings with the dense retrieval approach in your previous FAISS notebook.

**Learning style:** Run each cell, examine the output, then answer the **Think** questions before continuing.

**Dataset:** The same 10 z/OS documents from `01_embeddings-checkpoint.ipynb` and `02_faiss_flat.ipynb`, plus optional extra documents for exact-code experiments. This notebook runs independently; no variables from earlier notebooks are required.

**Prerequisites:** `numpy`, `pandas`, `matplotlib`. Optional: `rank-bm25`, `sentence-transformers`. Install missing packages in your notebook environment with `%pip install numpy pandas matplotlib rank-bm25 sentence-transformers` (the embedding model may download on first use).

## Lab 0 — Corpus and tokenization
A *tokenizer* decides which sequences count as terms. For enterprise documents, that decision matters: `IEC030I`, `B37-04`, and `SYS1.PARMLIB` should remain searchable identifiers. Here we deliberately use simple lowercase whitespace/punctuation trimming, **not** a production tokenizer.

In [ ]:
import re
import math
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base_documents = [
    "JES2 spool utilization is increasing due to retained job output.",
    "High JES2 spool usage can delay batch job processing.",
    "Batch workload performance is affected by resource contention.",
    "JES2 spool cleanup helps reclaim disk capacity.",
    "CICS transaction response time is increasing.",
    "Db2 query optimization improves database performance.",
    "TCP/IP network configuration on z/OS.",
    "Mainframe CPU utilization is high during peak workload.",
    "Storage shortages can cause batch job failures.",
    "Use SDSF to monitor JES2 spool and job output."
]

extra_documents = [
    "IEC030I B37-04 indicates a dataset space allocation problem.",
    "Check SYS1.PARMLIB configuration for z/OS system settings.",
    "A B37-04 abend may indicate insufficient dataset space.",
    "WLM service classes prioritize work under CPU resource contention."
]

# Set True to add four identifier-focused docs to the original ten.
USE_EXTRAS = False
documents = base_documents + (extra_documents if USE_EXTRAS else [])
ids = [f"D{i+1}" for i in range(len(documents))]

def tokenize(text):
    # Keep embedded '/', '.' and '-' so technical identifiers survive.
    return [t.strip('.,;:!?()[]{}"\'').lower() for t in text.split() if t.strip('.,;:!?()[]{}"\'')]

tokenized_docs = [tokenize(d) for d in documents]
df_docs = pd.DataFrame({"id": ids, "text": documents, "tokens": tokenized_docs})
display(df_docs)
print("Document count:", len(documents))
print("Example tokens:", tokenize("What does IEC030I B37-04 mean in SYS1.PARMLIB?"))

**Think:** What might go wrong if a tokenizer splits `B37-04` into `B37` and `04`? What should happen to `JES2` vs `jes2`?

## Lab 1 — Build an inverted index manually
An inverted index maps **term → posting list of documents**. A posting can also include TF or term positions. Start with term → document ID lists.

In [ ]:
inverted_index = defaultdict(list)
for doc_id, tokens in zip(ids, tokenized_docs):
    for term in sorted(set(tokens)):
        inverted_index[term].append(doc_id)

for term in ["jes2", "spool", "batch", "wlm", "iec030i"]:
    print(f"{term:10s} -> {inverted_index.get(term, [])}")

def and_search(query):
    terms = tokenize(query)
    if not terms:
        return []
    postings = [set(inverted_index.get(t, [])) for t in terms]
    return sorted(set.intersection(*postings), key=lambda x: int(x[1:]))

print('AND search "JES2 spool":', and_search("JES2 spool"))
print('AND search "JES2 batch":', and_search("JES2 batch"))

**Exercise:** Implement `or_search(query)` by taking the **union** of posting lists instead of the intersection. Why might OR retrieval be useful before ranking?

In [ ]:
def or_search(query):
    # TODO: implement OR retrieval using set.union
    raise NotImplementedError("Implement OR retrieval, then remove this exception")

# Example expected for "JES2 spool": documents containing either term.
# print(or_search("JES2 spool"))

## Lab 2 — Term Frequency (TF)
Raw TF counts how often a term appears in one document. Higher TF is not always better; repeating a term 100 times should not give 100× the evidence.

In [ ]:
term_counts = [Counter(tokens) for tokens in tokenized_docs]

def tf(term, doc_index):
    return term_counts[doc_index][term.lower()]

for term in ["jes2", "spool", "job", "batch"]:
    print(term, {doc_id: tf(term, i) for i, doc_id in enumerate(ids) if tf(term, i)})

# Explore diminishing returns when TF gets large.
frequencies = np.arange(0, 21)
plt.figure(figsize=(7, 3))
plt.plot(frequencies, frequencies, label="Raw TF")
plt.xlabel("Occurrences of term in document")
plt.ylabel("Raw TF")
plt.title("Raw term frequency rises linearly")
plt.legend()
plt.show()

**Think:** If one long log file repeats `spool` 50 times, should it always outrank a focused diagnostic paragraph containing `spool` twice?

## Lab 3 — Document Frequency (DF) and Inverse Document Frequency (IDF)
DF counts **documents containing a term**, not its total repetitions. A rare identifier can be more discriminative than a widespread word. We compare *classic* IDF and a common nonnegative BM25 IDF variant. They are **not interchangeable formulas**.

In [ ]:
N = len(documents)
df_terms = Counter(term for tokens in tokenized_docs for term in set(tokens))

def classic_idf(term):
    df = df_terms[term.lower()]
    return math.log(N / df) if df else float('nan')

def bm25_idf(term):
    df = df_terms[term.lower()]
    return math.log(1 + (N - df + 0.5) / (df + 0.5))

idfs = pd.DataFrame([
    {"term": t, "DF": df_terms[t], "classic_IDF": round(classic_idf(t), 3),
     "BM25_IDF_variant": round(bm25_idf(t), 3)}
    for t in sorted(df_terms)
]).sort_values(["DF", "term"])
display(idfs.head(15))
display(idfs.tail(10))

**Exercise:** Find the most common term and the rarest terms. Does BM25 IDF stay nonnegative? What happens for query terms absent from the entire corpus?

## Lab 4 — Build a basic TF-IDF search baseline
Here we score a query with the sum of `raw TF × classic IDF` over its unique terms. This is a **simplified learning baseline**, not scikit-learn's L2-normalized vector-space TF-IDF implementation.

In [ ]:
def tfidf_scores(query):
    terms = set(tokenize(query))
    return np.array([
        sum(tf(t, i) * classic_idf(t) for t in terms if df_terms[t] > 0)
        for i in range(N)
    ], dtype=float)

def display_results(scores, k=5):
    order = np.argsort(-np.asarray(scores), kind="stable")[:min(k, len(scores))]
    return pd.DataFrame([
        {"rank": rank, "document_id": ids[i], "score": round(float(scores[i]), 4), "document": documents[i]}
        for rank, i in enumerate(order, 1)
    ])

display(display_results(tfidf_scores("JES2 spool batch")))

## Lab 5 — Implement Okapi-style BM25 from scratch
For each query term, BM25 multiplies IDF by a *saturating TF factor* and a *length adjustment*.

\[
\mathrm{BM25}(D,Q)=\sum_{t\in Q}\mathrm{IDF}(t)
rac{f(t,D)(k_1+1)}{f(t,D)+k_1\left(1-b+b
rac{|D|}{\mathrm{avgdl}}
ight)}
\]

- `k1`: controls TF saturation; common starting point 1.2–2.0.
- `b`: controls length normalization; common starting point 0.75.
- `|D|`: token count in the current document.
- `avgdl`: average document token count.

We use the **nonnegative BM25 IDF variant from Lab 3**. Libraries such as `rank-bm25` may use a different IDF convention, so do not expect identical absolute scores.

In [ ]:
doc_lengths = np.array([len(tokens) for tokens in tokenized_docs])
avgdl = float(doc_lengths.mean())

def bm25_scores(query, k1=1.5, b=0.75):
    query_terms = tokenize(query)
    scores = np.zeros(N, dtype=float)
    for term in query_terms:
        if df_terms[term] == 0:
            continue
        idf = bm25_idf(term)
        for i in range(N):
            frequency = tf(term, i)
            denominator = frequency + k1 * (1 - b + b * doc_lengths[i] / avgdl)
            scores[i] += idf * frequency * (k1 + 1) / denominator
    return scores

query = "JES2 spool batch"
display(display_results(bm25_scores(query), k=5))
print("avgdl:", round(avgdl, 2), "| doc lengths:", doc_lengths.tolist())

## Lab 6 — Visualize TF saturation and document length
Control one factor at a time. Use the same IDF (=1) to focus on the BM25 TF term. Then compare short vs long documents at a fixed term frequency.

In [ ]:
k1 = 1.5
freq = np.arange(0, 31)

def tf_factor(f, length_ratio=1.0, k1=1.5, b=0.75):
    return f * (k1 + 1) / (f + k1 * (1 - b + b * length_ratio))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(freq, freq, label="Raw TF")
axes[0].plot(freq, tf_factor(freq), label="BM25 TF factor")
axes[0].set(title="Diminishing returns", xlabel="TF", ylabel="Contribution (IDF=1)")
axes[0].legend()

ratios = np.linspace(0.25, 5, 100)
axes[1].plot(ratios, tf_factor(3, ratios, b=0), label="b=0 (no normalization)")
axes[1].plot(ratios, tf_factor(3, ratios, b=0.75), label="b=0.75")
axes[1].plot(ratios, tf_factor(3, ratios, b=1), label="b=1")
axes[1].set(title="Length normalization (TF=3)", xlabel="Document length / average length", ylabel="Contribution (IDF=1)")
axes[1].legend()
plt.tight_layout()
plt.show()

**Exercises:** Change `k1` to 0.5 and 3.0; what happens to saturation? Change `b` between 0 and 1; which documents benefit from smaller length?

## Lab 7 — Compare TF-IDF and BM25 rankings
The *same query and same corpus* are critical for fair comparisons. Do not compare raw BM25 scores with raw cosine scores; compare **rank positions** and **relevance labels**.

In [ ]:
queries = [
    "Why are my batch jobs running slowly?",
    "JES2 spool job output",
    "Resource contention during batch processing",
    "TCP/IP network configuration",
    "IEC030I B37-04"
]

for q in queries:
    print("\nQUERY:", q)
    print("TF-IDF:", display_results(tfidf_scores(q), 3)["document_id"].tolist())
    print("BM25:  ", display_results(bm25_scores(q), 3)["document_id"].tolist())

print("\nNote: exact-code query matches only when USE_EXTRAS=True (rerun notebook).")

## Lab 8 — Compare with dense embedding retrieval (optional)
This cell requires `sentence-transformers` and may need to download `all-MiniLM-L6-v2`. It uses the **same embedding model** as your previous notebooks. If the package or model is unavailable, it prints a clear skip message and the rest of the lab remains usable.

In [ ]:
dense_scores_fn = None
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    dense_matrix = model.encode(documents, normalize_embeddings=True)
    def dense_scores_fn(q):
        q_vec = model.encode(q, normalize_embeddings=True)
        return dense_matrix @ q_vec
    print("Dense model loaded. Document matrix shape:", dense_matrix.shape)
except Exception as exc:
    print("Dense retrieval skipped:", type(exc).__name__, str(exc)[:240])

In [ ]:
for q in queries:
    print("\nQUERY:", q)
    print("BM25:", display_results(bm25_scores(q), 3)["document_id"].tolist())
    if dense_scores_fn is not None:
        print("Dense:", display_results(dense_scores_fn(q), 3)["document_id"].tolist())
    else:
        print("Dense: skipped")

**Think:** Why can a dense model retrieve a passage about WLM even when the query uses different wording? Why can BM25 be better for `IEC030I`? Do not assume one winner without relevance judgments.

## Lab 9 — Evaluate with labeled relevance
Use the labels from your earlier notebook for the batch-slowdown query: relevant documents **D2 and D3**. Labels are a *human judgment*, not derived from similarity scores. Extend to more labeled queries before drawing strong conclusions.

In [ ]:
def precision_recall_at_k(scores, relevant_ids, k=3):
    predicted = display_results(scores, k)["document_id"].tolist()
    relevant = set(relevant_ids)
    hits = len(set(predicted) & relevant)
    return {"retrieved": predicted, "precision@k": hits / k,
            "recall@k": hits / len(relevant) if relevant else float('nan')}

q = "Why are my batch jobs running slowly?"
labels = {"D2", "D3"}
print("BM25:", precision_recall_at_k(bm25_scores(q), labels, 3))
print("TF-IDF:", precision_recall_at_k(tfidf_scores(q), labels, 3))
if dense_scores_fn is not None:
    print("Dense:", precision_recall_at_k(dense_scores_fn(q), labels, 3))

## Lab 10 — Stretch: Reciprocal Rank Fusion (RRF)
RRF merges **rankings**, not incomparable raw scores. Formula: `RRF(d) = Σ 1 / (k + rank_system(d))`; a typical value is `k=60`. Compare BM25 and dense rankings after you have dense search working.

In [ ]:
def rrf_fuse(score_arrays, rrf_k=60):
    fused = np.zeros(N, dtype=float)
    for arr in score_arrays:
        ordering = np.argsort(-np.asarray(arr), kind="stable")
        for rank, doc_index in enumerate(ordering, start=1):
            fused[doc_index] += 1 / (rrf_k + rank)
    return fused

q = "JES2 spool batch jobs"
inputs = [bm25_scores(q)]
if dense_scores_fn is not None:
    inputs.append(dense_scores_fn(q))
    display(display_results(rrf_fuse(inputs), 5))
else:
    print("Run the dense model cell to compare genuine hybrid retrieval.")

## Checkpoint — Answer in your own words

1. How does an inverted index differ from scanning every document?
2. What is DF vs TF? Why does a rare term receive higher IDF?
3. Why is `raw TF × IDF` not the same as BM25?
4. What does `k1` change? What does `b` change?
5. Why can BM25 miss a paraphrase? Why can dense retrieval miss an exact error code?
6. Why should you evaluate retrieval using relevant-document labels instead of raw similarity scores?
7. How does RRF avoid directly comparing BM25 and cosine score scales?

**Next lab:** Hybrid Search + RRF using a larger labeled query set, then MRR and nDCG@K.

**Production note:** This lab uses a simple tokenizer and an educational BM25 implementation. Production lexical engines have richer analyzers, incremental indexing, metadata/ACL filtering, and operational capabilities.